# ACV: train and evaluate balanced logistic regression (C=0.1)

Run the cells in order, or select **Run All**. This notebook loads the raw Train dataset, builds features, trains new models, and prints the measured scores. All implementation is in this notebook. No saved model or result file is required.

**Local data:** copy `ACV` from the supplied `PS3/02_Datasets` bundle into this folder's `data/`, retaining the Train names below. Data stays local and is ignored by Git.

```text
ACV/
  train.ipynb
  data/
    Train_Labels.csv
    Train/
      acv_case_01.xlsx ... acv_case_06.xlsx
```

Use Python 3.11. Install the pinned CPU libraries once in the notebook's Python environment:
```python
%pip install numpy==1.26.4 pandas==2.2.1 scipy==1.12.0 scikit-learn==1.4.1.post1 openpyxl==3.1.5 rainflow==3.2.0 threadpoolctl==3.4.0
```

The displayed scores are cross-validation on labelled **Train** data. The recipe was selected in earlier experiments on this corpus, so these are exploratory validation results. Official Test is never read. There are only six cases across five physical trains; case 04 has missing shared measurements for some cars. Cabin/setpoint aliases are treated as equivalent units. A simple persistence baseline previously tied the learned model.

## 1. Imports and data location

In [1]:
from pathlib import Path
import sys, time, hashlib
import numpy as np
import pandas as pd
import sklearn
from IPython.display import display
from threadpoolctl import threadpool_limits
HERE = Path.cwd() if Path.cwd().name == 'ACV' else Path.cwd()/'ACV'
DATA = HERE/'data'
assert DATA.is_dir(), f'Place the ACV Train dataset in {DATA} first (see the cell above).'
SEED = 17
started = time.perf_counter()
print('Python:', sys.version.split()[0], '| NumPy:', np.__version__, '| pandas:', pd.__version__, '| sklearn:', sklearn.__version__)
print('Data:', DATA.resolve())
import re, warnings
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import LeaveOneOut, LeaveOneGroupOut

Python: 3.11.4 | NumPy: 1.26.4 | pandas: 2.2.1 | sklearn: 1.4.1.post1
Data: C:\000NebulaX\nebulax_p3\ACV\data


## 2. Extract each car’s cooling features
Compare cabin temperature with same-mode peers within each case. The fixed plausibility mask excludes sentinels. Imputation and scaling will be fitted using only the fitting cases.

In [2]:
ALIASES = {'indoor': ['Indoor Average Temperature', 'Passenger Cabin Temperature Detected Value'], 'target': ['ACV Control Temperature (Cooling)', 'Target Temperature Value'], 'mode': ['ACV Running Mode'], 'valid': ['ACV Information Valid']}

def feature_case(path):
    df = pd.read_excel(path, engine='openpyxl', dtype=object)
    headers = {}
    for c in df.columns:
        match = re.fullmatch(r'Car (\d+) - (.+)', str(c))
        if match:
            headers.setdefault(match[1], {})[match[2]] = c
    cars = sorted(headers)
    assert len(cars) == 8
    n = len(df)
    indoor = np.full((n, 8), np.nan)
    target = np.full((n, 8), np.nan)
    modes = np.empty((n, 8), dtype=object)
    valid = np.ones((n, 8), dtype=bool)
    source_map = {}
    for j, car in enumerate(cars):
        source_map[car] = {}
        for semantic, aliases in ALIASES.items():
            col = next((headers[car][x] for x in aliases if x in headers[car]), None)
            source_map[car][semantic] = col
            if semantic in ('indoor', 'target'):
                values = pd.to_numeric(df[col], errors='coerce').to_numpy(dtype=float) if col else np.full(n, np.nan)
                # Fixed plausibility mask; -50 sentinels are excluded. No labels used.
                values[(values < -20) | (values > 70)] = np.nan
                (indoor if semantic == 'indoor' else target)[:, j] = values
            elif semantic == 'mode':
                modes[:, j] = df[col].fillna('').astype(str).str.lower().to_numpy() if col else ''
            elif col:
                valid[:, j] = df[col].fillna('').astype(str).str.lower().eq('valid').to_numpy()
    cooling = np.vectorize(lambda s: 'cooling' in s)(modes)
    valid &= np.isfinite(indoor)
    excess = indoor-target
    rows = []
    qc = []
    for j, car in enumerate(cars):
        mask = valid[:, j] & cooling[:, j]
        others = [k for k in range(8) if k != j]
        peers = indoor[:, others].copy()
        comparable = valid[:, others] & cooling[:, others] & (modes[:, others] == modes[:, j, None])
        peers[~comparable] = np.nan
        with warnings.catch_warnings():
            warnings.simplefilter('ignore', RuntimeWarning)
            peer = np.nanmedian(peers, axis=1)
        residual = indoor[:, j]-peer
        residual[~mask | (comparable.sum(axis=1) < 2)] = np.nan
        good = residual[np.isfinite(residual)]
        errors = excess[mask, j]
        errors = errors[np.isfinite(errors)]
        rows.append([
            np.median(good) if good.size else np.nan,
            np.quantile(good, .9) if good.size else np.nan,
            np.mean(good > 1) if good.size else np.nan,
            np.median(errors) if errors.size else np.nan,
            np.quantile(errors, .9) if errors.size else np.nan,
            float(np.mean(cooling[:, j])), float(np.mean(valid[:, j])),
        ])
        qc.append({'car':car, 'rows':n, 'valid_cooling_rows':int(mask.sum()), 'comparable_peer_rows':int(good.size)})
    train_id = str(df['Train number'].dropna().iloc[0])
    return np.array(rows), cars, train_id, {'columns_used': source_map, 'cars': qc}

def rank_case(cars, scores):
    return [cars[i] for i in sorted(range(len(cars)), key=lambda i: (-float(scores[i]), cars[i]))]

## 3. Load all six raw training workbooks

In [3]:
labels=pd.read_csv(DATA/'Train_Labels.csv',dtype=str)
names=sorted(labels.filename.tolist())
truth=dict(zip(labels.filename,labels.faulty_car))
X,cars,trains=[],[],[]
for name in names:
    assert Path(name).name==name
    values,identifiers,train_id,quality=feature_case(DATA/'Train'/name)
    X.append(values);cars.append(identifiers);trains.append(train_id)
    print(f'{name}: {values.shape[0]} cars, {values.shape[1]} features, train {train_id}')
y=[np.array([int(c==truth[n]) for c in ids]) for n,ids in zip(names,cars)]
print('Cases:',len(names),'| Physical trains:',len(set(trains)))

acv_case_01.xlsx: 8 cars, 7 features, train 0620


acv_case_02.xlsx: 8 cars, 7 features, train 0620


acv_case_03.xlsx: 8 cars, 7 features, train 0619


acv_case_04.xlsx: 8 cars, 7 features, train 0208


acv_case_05.xlsx: 8 cars, 7 features, train 0407


acv_case_06.xlsx: 8 cars, 7 features, train 0408
Cases: 6 | Physical trains: 5


## 4. Fit and validate by whole case and physical train

In [4]:
def fit_model(indices):
    raw=np.concatenate([X[i] for i in indices])
    preprocessing=make_pipeline(SimpleImputer(strategy='median',keep_empty_features=True),StandardScaler())
    scaled=preprocessing.fit_transform(raw)
    model=LogisticRegression(C=.1,class_weight='balanced',solver='liblinear',random_state=SEED,max_iter=3000)
    model.fit(scaled,np.concatenate([y[i] for i in indices]))
    return preprocessing,model

def ranking(fitted,i):
    preprocessing,model=fitted
    return rank_case(cars[i],model.decision_function(preprocessing.transform(X[i])))

def rank_score(i,ranked):
    return (len(ranked)-ranked.index(truth[names[i]]))/len(ranked)

protocols={'leave_case_out':LeaveOneOut().split(names),
           'leave_train_out':LeaveOneGroupOut().split(names,groups=trains)}
fold_rows,case_rows=[],[]
for protocol,partitions in protocols.items():
    for fold,(fit,val) in enumerate(partitions,1):
        assert not set(fit)&set(val)
        if protocol=='leave_train_out': assert not {trains[i] for i in fit}&{trains[i] for i in val}
        with threadpool_limits(limits=2): fitted=fit_model(fit)
        fit_score=np.mean([rank_score(i,ranking(fitted,i)) for i in fit])
        val_scores=[]
        for i in val:
            ranked=ranking(fitted,i);value=rank_score(i,ranked);val_scores.append(value)
            case_rows.append({'protocol':protocol,'file':names[i],'true_car':truth[names[i]],
                              'true_rank':ranked.index(truth[names[i]])+1,'score':value,'ranking':'|'.join(ranked)})
        fold_rows.append({'protocol':protocol,'fold':fold,'train_fit':fit_score,'validation':np.mean(val_scores)})
        print(f'{protocol}, fold {fold}: train={fit_score:.6f}, validation={np.mean(val_scores):.6f}')

leave_case_out, fold 1: train=0.950000, validation=1.000000


leave_case_out, fold 2: train=0.950000, validation=1.000000
leave_case_out, fold 3: train=0.950000, validation=1.000000
leave_case_out, fold 4: train=1.000000, validation=0.750000
leave_case_out, fold 5: train=0.950000, validation=1.000000
leave_case_out, fold 6: train=0.950000, validation=1.000000
leave_train_out, fold 1: train=1.000000, validation=0.750000


leave_train_out, fold 2: train=0.950000, validation=1.000000


leave_train_out, fold 3: train=0.950000, validation=1.000000
leave_train_out, fold 4: train=0.950000, validation=1.000000
leave_train_out, fold 5: train=0.937500, validation=1.000000


## 5. Display scores and train the final model on all Train cases

In [5]:
results=pd.DataFrame(case_rows)
display(results)
for protocol,rows in results.groupby('protocol'):
    print(f"{protocol}: rank-decay={rows.score.mean():.9f}, top-1={(rows.true_rank==1).mean():.6f}, top-3={(rows.true_rank<=3).mean():.6f}")
print('Random-ranking expected score: 0.562500000')
validation_score=results.loc[results.protocol=='leave_case_out','score'].mean()
train_score=pd.DataFrame(fold_rows).query("protocol=='leave_case_out'").train_fit.mean()
print(f'Mean fitting-fold score: {train_score:.9f}')
with threadpool_limits(limits=2): final_model=fit_model(np.arange(len(names)))
print('Final model trained on all six cases; available as final_model.')
print(f'Elapsed: {time.perf_counter()-started:.1f} seconds')

,protocol,file,true_car,true_rank,score,ranking
0,leave_case_out,acv_case_01.xlsx,01,1,1.00,01|08|05|06|07|03|04|02
1,leave_case_out,acv_case_02.xlsx,02,1,1.00,02|03|01|07|04|06|08|05
2,leave_case_out,acv_case_03.xlsx,03,1,1.00,03|04|08|02|01|07|05|06
3,leave_case_out,acv_case_04.xlsx,01,3,0.75,04|02|01|05|06|07|08|03
4,leave_case_out,acv_case_05.xlsx,04,1,1.00,04|02|03|06|07|01|08|05
5,leave_case_out,acv_case_06.xlsx,06,1,1.00,06|08|02|03|04|01|05|07
6,leave_train_out,acv_case_04.xlsx,01,3,0.75,04|02|01|05|06|07|08|03
7,leave_train_out,acv_case_05.xlsx,04,1,1.00,04|02|03|06|07|01|08|05
8,leave_train_out,acv_case_06.xlsx,06,1,1.00,06|08|02|03|04|01|05|07
9,leave_train_out,acv_case_03.xlsx,03,1,1.00,03|04|08|02|01|07|05|06


leave_case_out: rank-decay=0.958333333, top-1=0.833333, top-3=1.000000
leave_train_out: rank-decay=0.958333333, top-1=0.833333, top-3=1.000000
Random-ranking expected score: 0.562500000
Mean fitting-fold score: 0.958333333
Final model trained on all six cases; available as final_model.
Elapsed: 137.6 seconds
